In [ ]:
def select_features(
    df_full: pd.DataFrame,
    feature_cols: list[str],
    region_tag: str,
    cache_dir: Path,
    *,
    gap: int,
    horizon: int,
    force_rerun: bool = False,
    n_features: int = 200,
    corr_threshold: float = 0.90,
    test_months: int = 12,
    valid_months: int = 6,
) -> list[str]:
    """Select the most informative features for a region via mutual information.

    Results are cached to <cache_dir>/selected_features_<region_tag>.json so
    the expensive MI computation only runs once per region.
    """
    import json
    from sklearn.feature_selection import mutual_info_regression

    FORECAST_GAP     = gap
    FORECAST_HORIZON = horizon

    cache_file = Path(cache_dir) / f"selected_features_{region_tag}.json"

    if not force_rerun and cache_file.exists():
        with open(cache_file) as f:
            cached = json.load(f)
        cols = [c for c in cached if c in df_full.columns]
        if cols:
            print(f"  Loaded {len(cols)} cached features from {cache_file.name}", flush=True)
            return cols

    cutoff_test  = df_full.index[-1] - pd.DateOffset(months=test_months)
    cutoff_valid = cutoff_test - pd.DateOffset(months=valid_months)
    train_df = df_full[df_full.index <= cutoff_valid]

    if train_df.empty:
        print("  Warning: empty train split, falling back to all features", flush=True)
        return feature_cols

    null_frac  = train_df[feature_cols].isna().mean()
    candidates = [c for c in feature_cols if null_frac[c] <= 0.05]
    if len(candidates) < 30:
        candidates = list(feature_cols)

    X = train_df[candidates].fillna(0.0).values.astype(np.float32)

    # Subsample for MI — up to 100 k rows for a more reliable ranking on large datasets.
    rng = np.random.default_rng(42)
    n_mi = min(100_000, len(X))
    mi_idx = rng.choice(len(X), size=n_mi, replace=False)
    mi_idx.sort()
    X_mi = X[mi_idx]

    t1    = f"target_h{FORECAST_GAP}"
    tmid  = f"target_h{FORECAST_GAP + FORECAST_HORIZON // 2}"
    tlast = f"target_h{FORECAST_GAP + FORECAST_HORIZON - 1}"

    scores = np.zeros(len(candidates), dtype=np.float64)
    for tcol in [t1, tmid, tlast]:
        if tcol not in train_df.columns:
            continue
        y_full = train_df[tcol].fillna(train_df[tcol].median()).values.astype(np.float32)
        y_mi   = y_full[mi_idx]
        scores += mutual_info_regression(X_mi, y_mi, discrete_features=False, n_neighbors=3, random_state=42, n_jobs=-1)

    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)

    # Greedy correlation dedup: drop features too correlated with already-selected ones.
    # Vectorised: maintain a (n_selected, n_rows) matrix and use matmul for batch correlation.
    X_norm = X - X.mean(axis=0, keepdims=True)
    X_norm /= (X_norm.std(axis=0, keepdims=True) + 1e-8)
    col_to_idx = {c: i for i, c in enumerate(candidates)}
    selected: list[str] = []
    sel_mat: np.ndarray | None = None  # shape (n_selected, n_rows)
    k = min(n_features, len(candidates))

    for col, _score in ranked:
        if len(selected) >= k:
            break
        vec = X_norm[:, col_to_idx[col]]          # (n_rows,)
        if sel_mat is not None:
            corr = np.abs(sel_mat @ vec) / len(vec)  # (n_selected,) — batch dot product
            if corr.max() > corr_threshold:
                continue
            sel_mat = np.vstack([sel_mat, vec[np.newaxis, :]])
        else:
            sel_mat = vec[np.newaxis, :]
        selected.append(col)

    if not selected:
        selected = feature_cols[:k]

    cache_file.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_file, "w") as f:
        json.dump(selected, f, indent=2)
    print(f"  Selected {len(selected)} features (from {len(candidates)}) -> {cache_file.name}", flush=True)
    return selected
